# NCA-LM Unified Kaggle GPU Runner (Phase 1 & Phase 2)

This unified notebook executes the complete experimental suite on Kaggle GPU in a single session:
1. **Phase 1 Baselines:**
   - Primary Transformer (Full Causal Attention, ~10M)
   - Sliding-Window Transformer ($W=128$, ~10M, context-matched control)
   - Mamba Selective SSM Reference (~10M)
   - GRU Recurrent Baseline (~10M)
2. **Phase 2 2x2 Factorial Matrix:**
   - Variant A: Shared NCA Rule ($d=288, K=6$, ~3.6M) — Baseline cellular rule
   - Variant B: Unshared CNN Stack ($d=160, K=6$, ~3.6M) — Pure sharing control (equal params)
   - Variant C: Unshared CNN Stack ($d=288, K=6$, ~9.8M) — Width-matched control (d=288)
   - Variant D: Shared NCA Rule ($d=576, K=6$, ~9.7M) — Capacity compensation control

> **Prerequisite:** Set Accelerator to **GPU (T4 x1 or P100)** in the right sidebar.

In [ ]:
# Cell 1: Environment Setup & Repository Clone
import os, sys
!git clone https://github.com/Zenoguy/NCA-sim.git /kaggle/working/NCA-sim || (cd /kaggle/working/NCA-sim && git pull origin main)
%cd /kaggle/working/NCA-sim
!pip install -q tokenizers pyyaml

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Verify Groundwork & Data Splits (Phase 0 Floor: 3-Gram PPL = 89.56)
!python scripts/run_level0.py
!python scripts/run_synthetic_smoke.py

## Phase 1: Baseline Calibration Runs (~10M Parameter Budget)

In [ ]:
# Cell 3: Train Primary Transformer (Full Causal Attention Reference, ~10M)
!python train.py --config configs/level1_transformer.yaml

In [ ]:
# Cell 4: Train Sliding-Window Transformer (W=128 Control, ~10M)
!python train.py --config configs/level1_transformer_sliding.yaml

In [ ]:
# Cell 5: Train Mamba Selective SSM Baseline (~10M)
!python train.py --config configs/level1_mamba.yaml

In [ ]:
# Cell 6: Train GRU Recurrent Baseline (~10M)
!python train.py --config configs/level1_gru.yaml

In [ ]:
# Cell 7: Format & Display Phase 1 Calibration Table
!python scripts/run_level1.py --action table

## Phase 2: 2x2 Factorial Matrix Runs (Weight-Sharing & Capacity Compensation)

In [ ]:
# Cell 8: Train Variant A — Shared NCA Rule (d=288, K=6, ~3.6M)
!python train.py --config configs/level2_nca_shared_3m.yaml

In [ ]:
# Cell 9: Train Variant B — Unshared CNN Stack (d=160, K=6, ~3.6M, Pure Sharing Control)
!python train.py --config configs/level2_nca_unshared_3m.yaml

In [ ]:
# Cell 10: Train Variant C — Unshared CNN Stack (d=288, K=6, ~9.8M, Width Control)
!python train.py --config configs/level2_nca_unshared_10m.yaml

In [ ]:
# Cell 11: Train Variant D — Shared NCA Rule (d=576, K=6, ~9.7M, Capacity Compensation)
!python train.py --config configs/level2_nca_shared_10m.yaml

## Results Aggregation & Scientific Gate Evaluation

In [ ]:
# Cell 12: Generate Phase 2 Factorial Matrix & Evaluate Gates
!python scripts/run_level2.py --action table

# Archive outputs for easy download
!tar -czvf /kaggle/working/nca_lm_outputs.tar.gz outputs/
print("All results archived to /kaggle/working/nca_lm_outputs.tar.gz")